# Function 7: Black-box optimisation

This notebook documents 13 rounds of model fitting, candidate selection and evaluated outputs for Function 7. Alternative approaches are retained where they informed the weekly decisions.

**Main result:** GP optimisation improved the results through Week 12; identifying Hartmann-6 led to 3.322368 in Week 13.

**Usage:** Run the setup and then the weekly sections in order. The initial datasets are in this folder; the full query history is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv). Some early searches generate large candidate grids and may require substantial memory.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import differential_evolution
from scipy.spatial import cKDTree
from scipy.stats import truncnorm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF


def resolve_initial_path(filename: str) -> Path:
    """Find the initial dataset for Function 7."""
    for folder in (Path.cwd(), Path.cwd() / "function_7"):
        candidate = folder / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found. Place it in the function_7/ folder."
    )

## Reusable routines

The helper functions below handle standard GP fitting, UCB scoring and recording evaluated queries. The model settings and experiments remain visible within each week. Global/local candidate generation is shared across the early rounds.

In [ ]:
def fit_gp_model(X, y, **settings):
    """Fit a GP using the explicit kernel and settings chosen for a given week."""
    gp = GaussianProcessRegressor(**settings)
    gp.fit(X, y)
    return gp


def ucb_score(mean, std, beta):
    """Upper Confidence Bound: predictive mean plus weighted uncertainty."""
    return mean + beta * std


def add_evaluation(X, y, x_observed, y_observed):
    """Add one completed black-box evaluation to the training history."""
    x_row = np.asarray(x_observed, dtype=float).reshape(1, -1)
    y_value = np.asarray(y_observed, dtype=float).reshape(1)
    if x_row.shape[1] != X.shape[1]:
        raise ValueError("The evaluated input has the wrong number of dimensions.")
    return np.vstack([X, x_row]), np.concatenate([y, y_value])

def sample_hybrid_candidates(best_x, n_global, n_local, seed, *,
                             local_scale=0.08, local_method="truncated"):
    """Combine uniform global samples with local samples near the current best.

    The local method is either "truncated" (truncated normal, inside [0, 1])
    or "clipped" (normal samples clipped to [0, 1]). Random number generation
    follows the same sequence for a given seed as the weekly experiments.
    """
    best_x = np.asarray(best_x, dtype=float).reshape(-1)
    n_features = best_x.size
    rng = np.random.default_rng(seed)

    X_global = rng.uniform(0, 1, size=(n_global, n_features))
    if local_method == "clipped":
        X_local = rng.normal(
            loc=best_x, scale=local_scale, size=(n_local, n_features)
        )
        X_local = np.clip(X_local, 0, 1)
    elif local_method == "truncated":
        X_local = np.empty((n_local, n_features))
        for j in range(n_features):
            a = (0 - best_x[j]) / local_scale
            b = (1 - best_x[j]) / local_scale
            X_local[:, j] = truncnorm.rvs(
                a, b, loc=best_x[j], scale=local_scale,
                size=n_local, random_state=rng
            )
    else:
        raise ValueError("local_method must be 'clipped' or 'truncated'.")

    return np.round(np.vstack([X_global, X_local]), 6)


## Starting data

In [2]:
X = np.load(resolve_initial_path("initial_inputs.npy"))
y = np.load(resolve_initial_path("initial_outputs.npy"))

## Week 1

In [6]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

# UCB exploration parameter
beta = 1.96
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds="fixed"
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True
)
# Generate candidate inputs
# Define the search range for each input variable

x1_values = np.linspace(0, 1, 18)
x2_values = np.linspace(0, 1, 18)
x3_values = np.linspace(0, 1, 18)
x4_values = np.linspace(0, 1, 18)
x5_values = np.linspace(0, 1, 18)
x6_values = np.linspace(0, 1, 18)

# Generate every combination of x1 and x2
X1_grid, X2_grid, X3_grid, X4_grid, X5_grid, X6_grid = np.meshgrid(
    x1_values,
    x2_values,
    x3_values,
    x4_values,
    x5_values,
    x6_values
)

X_grid = np.column_stack([
    X1_grid.ravel(),
    X2_grid.ravel(),
    X3_grid.ravel(),
    X4_grid.ravel(),
    X5_grid.ravel(),
    X6_grid.ravel()
])
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_grid,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)
# Select the next query
best_index = np.argmax(UCB)

next_X = X_grid[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.05882353 0.52941176 0.23529412 0.23529412 0.41176471 0.70588235]

Predicted mean:
1.2267905216599737

Predicted standard deviation:
0.14387227828233276

UCB:
1.508780187093346


### Evaluated query

**Input:** `(0.058824, 0.529412, 0.235294, 0.235294, 0.411765, 0.705882)`  
**Output:** `1.3202039706195594`

In [ ]:
X_week1 = np.array([0.058824, 0.529412, 0.235294, 0.235294, 0.411765, 0.705882])
y_week1 = np.array([1.3202039706195594])
X_new, y_new = add_evaluation(X, y, X_week1, y_week1)

## Week 2

In [10]:
X = X_new
y = y_new

### Increase β to favour exploration

In [11]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 3

### Allow the kernel length scale to vary

In [12]:
# Configure the Gaussian Process
kernel = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.349)


### Use random candidates instead of a grid

In [13]:
# Generate candidate inputs
n_candidates = 1000000
n_features = 6

np.random.seed(42)

X_candidates = np.random.uniform(
    low=0,
    high=1,
    size=(n_candidates, n_features)
)

X_candidates = np.round(X_candidates, 6)

In [14]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [15]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.032179 0.466923 0.396815 0.027617 0.337565 0.685541]

Predicted mean:
1.2014897207886943

Predicted standard deviation:
0.22087289898038528

UCB:
1.8641084177298501


### Evaluated query

**Input:** `(0.032179, 0.466923, 0.396815, 0.027617, 0.337565, 0.685541)`  
**Output:** `1.196752992783604`

In [ ]:
X_week2 = np.array([0.032179, 0.466923, 0.396815, 0.027617, 0.337565, 0.685541])
y_week2 = np.array([1.196752992783604])
X_new, y_new = add_evaluation(X, y, X_week2, y_week2)

## Week 3

In [19]:
X = X_new
y = y_new

### Option 1: Isotropic GP with global/local sampling · selected

In [20]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 2.5

In [21]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.378)


In [22]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 6
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [23]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [24]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.011068 0.419633 0.035248 0.076911 0.321052 0.711068]

Predicted mean:
1.1790146367841394

Predicted standard deviation:
0.249076213912373

UCB:
1.801705171565072


### Option 2: Anisotropic GP with global/local sampling

In [25]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

beta = 2.5

In [26]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[0.999, 0.561, 2, 0.48, 0.224, 0.428])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [27]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 6
n_global = 800000
n_local = 200000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=45,
    local_scale=0.08,
    local_method="truncated",
)

In [28]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [29]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.020623 0.354277 0.124353 0.276585 0.329654 0.983962]

Predicted mean:
1.3175157841817864

Predicted standard deviation:
0.20189798457010477

UCB:
1.8222607456070483


### Evaluated query

**Input:** `(0.011068, 0.419633, 0.035248, 0.076911, 0.321052, 0.711068)`  
**Output:** `1.0505272787700064`

In [ ]:
X_week3 = np.array([0.011068, 0.419633, 0.035248, 0.076911, 0.321052, 0.711068])
y_week3 = np.array([1.0505272787700064])
X_new, y_new = add_evaluation(X, y, X_week3, y_week3)

## Week 4

In [33]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 1.0 · selected

In [34]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 1.0

In [35]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.383)


In [36]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 6
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [37]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [38]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[1.69000e-04 5.20979e-01 2.90541e-01 1.52532e-01 4.03023e-01 8.68106e-01]

Predicted mean:
1.3484213592008913

Predicted standard deviation:
0.1480060969786829

UCB:
1.4964274561795743


### Option 2: Anisotropic GP with β = 1.0 · alternative

In [39]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]

beta = 1.0

In [40]:
# Configure the Gaussian Process
kernel_aniso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.01, 2.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_aniso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_aniso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

Optimised kernel:
RBF(length_scale=[1.71, 0.728, 2, 0.51, 0.166, 0.397])


/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [41]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 6
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=46,
    local_scale=0.08,
    local_method="truncated",
)

In [42]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [43]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.006651 0.611334 0.864455 0.084049 0.419458 0.84548 ]

Predicted mean:
1.4796395507765685

Predicted standard deviation:
0.14631228858836393

UCB:
1.6259518393649324


### Evaluated query

**Input:** `(0.000169, 0.520979, 0.290541, 0.152532, 0.403023, 0.868106)`  
**Output:** `0.8485502161089783`

In [ ]:
X_week4 = np.array([0.000169, 0.520979, 0.290541, 0.152532, 0.403023, 0.868106])
y_week4 = np.array([0.8485502161089783])
X_new, y_new = add_evaluation(X, y, X_week4, y_week4)

## Week 5

In [47]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 1.0 · selected

In [48]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 1.0

In [49]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.29)


In [50]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 6
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=47,
    local_scale=0.08,
    local_method="truncated",
)

In [51]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [52]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.084936 0.380847 0.273011 0.167953 0.408335 0.586833]

Predicted mean:
1.5305856481172198

Predicted standard deviation:
0.16394612744874099

UCB:
1.6945317755659608


### Evaluated query

**Input:** `(0.084936, 0.380847, 0.273011, 0.167953, 0.408335, 0.586833)`  
**Output:** `1.7164922902311388`

In [ ]:
X_week5 = np.array([0.084936, 0.380847, 0.273011, 0.167953, 0.408335, 0.586833])
y_week5 = np.array([1.7164922902311388])
X_new, y_new = add_evaluation(X, y, X_week5, y_week5)

## Week 6

In [56]:
X = X_new
y = y_new

### Option 1: Isotropic GP with β = 0.5 · selected

In [57]:
# GP configuration
noise_assumption = 1e-10
rbf_lengthscale = 0.1

beta = 0.5

In [58]:
# Configure the Gaussian Process
kernel_iso = RBF(
    length_scale=rbf_lengthscale,
    length_scale_bounds=(0.05, 1.5)
)

model = fit_gp_model(
    X, y,
    kernel=kernel_iso,
    alpha=noise_assumption,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Initial kernel:")
print(kernel_iso)

print("\nOptimised kernel:")
print(model.kernel_)

Initial kernel:
RBF(length_scale=0.1)

Optimised kernel:
RBF(length_scale=0.346)


In [59]:
# Generate global and local candidate inputs
n_candidates = 1000000
n_features = 6
n_global = 900000
n_local = 100000
best_X = X[np.argmax(y)]
X_candidates = sample_hybrid_candidates(
    best_x=best_X,
    n_global=n_global,
    n_local=n_local,
    seed=48,
    local_scale=0.08,
    local_method="truncated",
)

In [60]:
# Predict candidate outputs
post_mean, post_std = model.predict(
    X_candidates,
    return_std=True
)
# Calculate UCB
UCB = ucb_score(post_mean, post_std, beta)

In [61]:
# Select the next query
best_index = np.argmax(UCB)

next_X = X_candidates[best_index]

print("Next input combination suggested by UCB:")
print(next_X)

print("\nPredicted mean:")
print(post_mean[best_index])

print("\nPredicted standard deviation:")
print(post_std[best_index])

print("\nUCB:")
print(UCB[best_index])

Next input combination suggested by UCB:
[0.069559 0.314593 0.303236 0.209135 0.430854 0.586598]

Predicted mean:
1.7338072082491491

Predicted standard deviation:
0.08227491890670492

UCB:
1.7749446677025016


### Evaluated query

**Input:** `(0.069559, 0.314593, 0.303236, 0.209135, 0.430854, 0.586598)`  
**Output:** `1.9009929205068512`

In [ ]:
X_week6 = np.array([0.069559, 0.314593, 0.303236, 0.209135, 0.430854, 0.586598])
y_week6 = np.array([1.9009929205068512])
X_new, y_new = add_evaluation(X, y, X_week6, y_week6)

## Week 7

In [65]:
# All observations through Week 6
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (36, 6)
assert y.shape == (36,)

# Tuned global GP
kernel = RBF(
    length_scale=0.3,
    length_scale_bounds=(0.02, 5.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

print("Optimised kernel:", model.kernel_)

# GP-UCB acquisition parameters
beta = 0.5
min_distance = 0.05
tree = cKDTree(X)

def negative_ucb(point):
    point = np.asarray(point)

    mu, std = model.predict(
        point.reshape(1, -1),
        return_std=True
    )

    distance = tree.query(point)[0]

    # Discourage near-duplicate queries
    penalty = 50 * max(
        0, min_distance - distance
    )

    return -(mu[0] + beta * std[0]) + penalty

# Optimise acquisition using multiple seeds
solutions = []

for seed in [42, 48, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 6,
        seed=seed,
        popsize=15,
        maxiter=180,
        tol=1e-8,
        polish=True
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError("No feasible candidate found.")

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)

mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 7 query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])

Optimised kernel: RBF(length_scale=0.384)
Week 7 query: [0.       0.22141  0.34079  0.290412 0.434461 0.598399]
Predicted mean: 2.0245420446285887
Predicted std: 0.08612008756188638
UCB: 2.067602088409532
Nearest distance: 0.1472754346963539


### Evaluated query

**Input:** `(0.000000, 0.221410, 0.340790, 0.290412, 0.434461, 0.598399)`  
**Output:** `2.0272493040297306`

In [ ]:
X_week7 = np.array([0. , 0.22141 , 0.34079 , 0.290412, 0.434461, 0.598399])
y_week7 = np.array([2.0272493040297306])
X_new, y_new = add_evaluation(X, y, X_week7, y_week7)

## Week 8

In [69]:
# All observations through Week 7
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (37, 6)
assert y.shape == (37,)

# Selected global GP
kernel = RBF(
    length_scale=0.4,
    length_scale_bounds=(0.02, 5.0)
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
)

print("Optimised kernel:", model.kernel_)

# Acquisition settings
beta = 0.5
min_distance = 0.05

tree = cKDTree(X)

def negative_ucb(points):
    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    distances = tree.query(candidates)[0]

    # Penalise points very close to previous queries
    penalty = 100 * np.maximum(
        0, min_distance - distances
    )

    scores = -(ucb_score(mu, std, beta)) + penalty

    return scores if points.ndim == 2 else scores[0]

# Global acquisition optimisation
solutions = []

for seed in [42, 59, 111]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 6,
        seed=seed,
        popsize=12,
        maxiter=190,
        tol=1e-7,
        updating="deferred",
        vectorized=True,
        polish=False
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError("No feasible candidate found.")

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(best_result.x, 6)

mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 8 query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("UCB:", mu[0] + beta * std[0])
print("Nearest distance:", tree.query(next_X)[0])

Optimised kernel: RBF(length_scale=0.417)
Week 8 query: [0.       0.213896 0.364995 0.24639  0.489427 0.58366 ]
Predicted mean: 2.021590939828275
Predicted std: 0.0768118572553286
UCB: 2.059996868455939
Nearest distance: 0.07628091492634317


### Evaluated query

**Input:** `(0.000000, 0.213896, 0.364995, 0.246390, 0.489427, 0.583660)`  
**Output:** `1.6088779246668488`

In [ ]:
X_week8 = np.array([0. , 0.213896, 0.364995, 0.24639 , 0.489427, 0.58366])
y_week8 = np.array([1.6088779246668488])
X_new, y_new = add_evaluation(X, y, X_week8, y_week8)

## Week 9

In [73]:
# Observations through Week 8
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (38, 6)
assert y.shape == (38,)
# Selected GP model
kernel = Matern(
    length_scale=0.4,
    length_scale_bounds=(0.02, 5.0),
    nu=1.5
)

model = fit_gp_model(
    X, y,
    kernel=kernel,
    alpha=1e-10,
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
)
print("Optimised kernel:", model.kernel_)
# Acquisition settings
beta = 0.5
min_distance = 0.10

tree = cKDTree(X)

def negative_ucb(points):
    points = np.asarray(points)

    candidates = (
        points.T if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = model.predict(
        candidates,
        return_std=True
    )

    distances = tree.query(candidates)[0]

    ucb = ucb_score(mu, std, beta)

    penalty = 100 * np.maximum(
        0,
        min_distance - distances
    )

    scores = -ucb + penalty

    return (
        scores if points.ndim == 2
        else scores[0]
    )
# Optimise GP-UCB
solutions = []

for seed in [42, 48, 59]:
    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 6,
        seed=seed,
        popsize=13,
        maxiter=170,
        tol=1e-7,
        vectorized=True,
        updating="deferred",
        polish=False
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible candidate found."
    )

best_result = min(
    solutions,
    key=lambda r: r.fun
)

next_X = np.round(best_result.x, 6)
# Report results
mu, std = model.predict(
    next_X.reshape(1, -1),
    return_std=True
)

print("Week 9 query:", next_X)
print("Predicted mean:", mu[0])
print("Predicted std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])
print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)
print("Current best:", y.max())

Optimised kernel: Matern(length_scale=0.498, nu=1.5)
Week 9 query: [1.00000e-06 2.50684e-01 2.91363e-01 3.14785e-01 3.41993e-01 6.12098e-01]
Predicted mean: 2.2155464376514105
Predicted std: 0.14175393628315933
GP-UCB: 2.28642340579299
Nearest observation distance: 0.11239231361618995
Current best: 2.0272493040297306


### Evaluated query

**Input:** `(0.000000, 0.250684, 0.291363, 0.314785, 0.341993, 0.612098)`  
**Output:** `2.425431565983359`

In [ ]:
X_week9 = np.array([0. , 0.250684, 0.291363, 0.314785, 0.341993, 0.612098])
y_week9 = np.array([2.425431565983359])
X_new, y_new = add_evaluation(X, y, X_week9, y_week9)

## Week 10

In [77]:
# Observations through Week 9
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (39, 6)
assert y.shape == (39,)
# Fit both Gaussian Process models
kernels = [
    RBF(
        length_scale=0.4,
        length_scale_bounds=(0.02, 5.0)
    ),
    Matern(
        length_scale=0.4,
        length_scale_bounds=(0.02, 5.0),
        nu=1.5
    )
]

models = []

for kernel in kernels:
    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-10,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    gp.fit(X, y)
    models.append(gp)

    print("Fitted kernel:", gp.kernel_)
# Ensemble predictions
weights = np.array([0.5, 0.5])

def ensemble_predict(points):

    predictions = [
        gp.predict(points, return_std=True)
        for gp in models
    ]

    means = np.array([
        prediction[0]
        for prediction in predictions
    ])

    variances = np.array([
        prediction[1]**2
        for prediction in predictions
    ])

    ensemble_mean = np.average(
        means,
        axis=0,
        weights=weights
    )

    ensemble_variance = np.average(
        variances + (
            means - ensemble_mean
        )**2,
        axis=0,
        weights=weights
    )

    return (
        ensemble_mean,
        np.sqrt(ensemble_variance)
    )
# Acquisition settings
beta = 0.5
min_distance = 0.10

tree = cKDTree(X)

def negative_ucb(points):

    points = np.asarray(points)

    candidates = (
        points.T
        if points.ndim == 2
        else points.reshape(1, -1)
    )

    mu, std = ensemble_predict(candidates)

    distances = tree.query(candidates)[0]

    ucb = ucb_score(mu, std, beta)

    # Margin helps ensure that rounding the
    # selected point preserves the constraint.
    penalty = 100 * np.maximum(
        0,
        min_distance + 0.001 - distances
    )

    scores = -ucb + penalty

    return (
        scores
        if points.ndim == 2
        else scores[0]
    )
# Optimise ensemble GP-UCB
solutions = []

for seed in [42, 59, 111]:

    result = differential_evolution(
        negative_ucb,
        bounds=[(0, 1)] * 6,
        seed=seed,
        popsize=14,
        maxiter=170,
        tol=1e-8,
        vectorized=True,
        updating="deferred",
        polish=True
    )

    if tree.query(result.x)[0] >= min_distance:
        solutions.append(result)

if not solutions:
    raise RuntimeError(
        "No feasible acquisition candidate found."
    )

best_result = min(
    solutions,
    key=lambda result: result.fun
)

next_X = np.round(
    best_result.x,
    6
)
# Report recommendation
mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

print("Week 10 query:", next_X)
print("Predicted mean:", mu[0])
print("Ensemble std:", std[0])
print("GP-UCB:", mu[0] + beta * std[0])
print(
    "Nearest observation distance:",
    tree.query(next_X)[0]
)
print("Current best:", np.max(y))

Fitted kernel: RBF(length_scale=0.335)
Fitted kernel: Matern(length_scale=0.571, nu=1.5)
Week 10 query: [0.016725 0.215301 0.249637 0.368315 0.28083  0.630035]
Predicted mean: 2.429223129807763
Ensemble std: 0.14582204473830415
GP-UCB: 2.5021341521769154
Nearest observation distance: 0.10099926152205273
Current best: 2.425431565983359


### Evaluated query

**Input:** `(0.016725, 0.215301, 0.249637, 0.368315, 0.280830, 0.630035)`  
**Output:** `2.3771077041323427`

In [ ]:
X_week10 = np.array([0.016725, 0.215301, 0.249637, 0.368315, 0.28083 , 0.630035])
y_week10 = np.array([2.3771077041323427])
X_new, y_new = add_evaluation(X, y, X_week10, y_week10)

## Week 11

In [80]:
# Observations through Week 10
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (40, 6)
assert y.shape == (40,)

best_idx = np.argmax(y)
best_X = X[best_idx]
best_y = y[best_idx]

print("Current best input:", best_X)
print("Current best output:", best_y)
# Fit six complementary GP models
models = []

for anisotropic in [False, True]:

    length_scale = (
        np.full(6, 0.4)
        if anisotropic else 0.4
    )

    kernels = [
        RBF(
            length_scale=length_scale,
            length_scale_bounds=(0.02, 6.0)
        ),
        Matern(
            length_scale=length_scale,
            length_scale_bounds=(0.02, 6.0),
            nu=1.5
        ),
        Matern(
            length_scale=length_scale,
            length_scale_bounds=(0.02, 6.0),
            nu=2.5
        )
    ]

    for kernel in kernels:

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-10,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=42
        )

        gp.fit(X, y)
        models.append(gp)

        print("Fitted kernel:", gp.kernel_)
# Ensemble prediction
def ensemble_predict(points):

    points = np.atleast_2d(points)

    predictions = [
        gp.predict(points, return_std=True)
        for gp in models
    ]

    means = np.array([
        prediction[0]
        for prediction in predictions
    ])

    variances = np.array([
        prediction[1]**2
        for prediction in predictions
    ])

    ensemble_mean = means.mean(axis=0)

    ensemble_variance = np.mean(
        variances + means**2,
        axis=0
    ) - ensemble_mean**2

    return (
        ensemble_mean,
        np.sqrt(np.maximum(ensemble_variance, 0))
    )
# Local search region
bounds = [
    (0.00, 0.12),
    (0.14, 0.36),
    (0.18, 0.44),
    (0.20, 0.44),
    (0.20, 0.48),
    (0.53, 0.72)
]

tree = cKDTree(X)

beta = 0.0
target_distance = 0.04
min_distance = 0.035
# Acquisition function
def negative_acquisition(point):

    mu, std = ensemble_predict(
        np.asarray(point).reshape(1, -1)
    )

    distance = tree.query(point)[0]

    acquisition = mu[0] + beta * std[0]

    penalty = 120 * max(
        0,
        target_distance - distance
    )**2

    return -acquisition + penalty
# Optimise acquisition
result = differential_evolution(
    negative_acquisition,
    bounds=bounds,
    seed=42,
    popsize=8,
    maxiter=95,
    tol=1e-6,
    polish=True
)

next_X = np.round(result.x, 6)

nearest_distance = tree.query(next_X)[0]

assert nearest_distance >= min_distance
# Evaluate candidate
mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

print("\nWeek 11 proposed input:")
print(next_X)

print("\nEnsemble predicted mean:", mu[0])
print("Ensemble predicted std:", std[0])
print("Predicted improvement:", mu[0] - best_y)
print("Nearest observation distance:", nearest_distance)

print("\nIndividual model predictions:")

for gp in models:
    prediction = gp.predict(next_X.reshape(1, -1))[0]
    print(gp.kernel_, ":", prediction)

print("\nSubmission format:")
print(" ".join(f"{value:.6f}" for value in next_X))

Current best input: [0.       0.250684 0.291363 0.314785 0.341993 0.612098]
Current best output: 2.425431565983359
Fitted kernel: RBF(length_scale=0.336)
Fitted kernel: Matern(length_scale=0.635, nu=1.5)
Fitted kernel: Matern(length_scale=0.496, nu=2.5)
Fitted kernel: RBF(length_scale=[0.726, 0.322, 0.964, 0.341, 0.275, 0.455])
Fitted kernel: Matern(length_scale=[1.21, 0.715, 1.89, 0.661, 0.429, 0.628], nu=1.5)
Fitted kernel: Matern(length_scale=[1.01, 0.515, 1.35, 0.521, 0.362, 0.545], nu=2.5)

Week 11 proposed input:
[0.       0.257096 0.291219 0.339459 0.312359 0.618491]

Ensemble predicted mean: 2.4525919919624672
Ensemble predicted std: 0.03659625209229135
Predicted improvement: 0.027160425979108282
Nearest observation distance: 0.03961041732928348

Individual model predictions:
RBF(length_scale=0.336) : 2.4699652640897596
Matern(length_scale=0.635, nu=1.5) : 2.4403354319147414
Matern(length_scale=0.496, nu=2.5) : 2.4527316100960195
RBF(length_scale=[0.726, 0.322, 0.964, 0.341, 0.

### Evaluated query

**Input:** `(0.000000, 0.257096, 0.291219, 0.339459, 0.312359, 0.618491)`  
**Output:** `2.4212223670576587`

In [ ]:
X_week11 = np.array([0., 0.257096, 0.291219, 0.339459, 0.312359, 0.618491])
y_week11 = np.array([2.4212223670576587])
X_new, y_new = add_evaluation(X, y, X_week11, y_week11)

## Week 12

In [83]:
# Observations through Week 11
X = np.asarray(X_new, dtype=float)
y = np.asarray(y_new, dtype=float).ravel()

assert X.shape == (41, 6)
assert y.shape == (41,)

best_idx = np.argmax(y)
best_X = X[best_idx]
best_y = y[best_idx]

print("Current best input:", best_X)
print("Current best output:", best_y)
# Fit six GP models
models = []

for anisotropic in [False, True]:

    length_scale = (
        np.full(6, 0.4)
        if anisotropic else 0.4
    )

    kernels = {
        "RBF": RBF(
            length_scale,
            length_scale_bounds=(0.02, 6.0)
        ),
        "Matern 3/2": Matern(
            length_scale,
            length_scale_bounds=(0.02, 6.0),
            nu=1.5
        ),
        "Matern 5/2": Matern(
            length_scale,
            length_scale_bounds=(0.02, 6.0),
            nu=2.5
        )
    }

    for name, kernel in kernels.items():

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-8,
            normalize_y=True,
            n_restarts_optimizer=3,
            random_state=42
        )

        gp.fit(X, y)

        label = name + (
            " ARD" if anisotropic else " isotropic"
        )

        models.append((label, gp))

        print(label, gp.kernel_)
# Ensemble predictions
def ensemble_predict(points):

    points = np.atleast_2d(points)

    means = []
    variances = []

    for _, gp in models:
        mu, std = gp.predict(
            points,
            return_std=True
        )

        means.append(mu)
        variances.append(std**2)

    means = np.asarray(means)
    variances = np.asarray(variances)

    ensemble_mean = means.mean(axis=0)

    ensemble_variance = (
        np.mean(variances + means**2, axis=0)
        - ensemble_mean**2
    )

    return (
        ensemble_mean,
        np.sqrt(np.maximum(ensemble_variance, 0))
    )
# Local search around promising region
bounds = [
    (0.00, 0.08),
    (0.16, 0.36),
    (0.19, 0.40),
    (0.22, 0.43),
    (0.20, 0.47),
    (0.52, 0.72)
]

tree = cKDTree(X)
min_distance = 0.02
# Exploitation-focused acquisition
def negative_acquisition(point):

    mu, std = ensemble_predict(
        np.asarray(point).reshape(1, -1)
    )

    distance = tree.query(point)[0]

    penalty = 200 * max(
        0,
        min_distance - distance
    )**2

    return -mu[0] + penalty
# Optimise candidate
result = differential_evolution(
    negative_acquisition,
    bounds=bounds,
    seed=42,
    popsize=10,
    maxiter=80,
    tol=1e-7,
    polish=True
)

optimised_X = np.round(result.x, 6)

print("\nOptimised candidate:", optimised_X)
# Evaluate recommended Week 12 point
next_X = np.array([
    0.000000,
    0.218122,
    0.253681,
    0.313223,
    0.316940,
    0.621175
])

mu, std = ensemble_predict(
    next_X.reshape(1, -1)
)

nearest_distance = tree.query(next_X)[0]

print("\nWeek 12 recommended input:")
print(" ".join(f"{v:.6f}" for v in next_X))

print("Ensemble predicted mean:", mu[0])
print("Ensemble predicted std:", std[0])
print("Predicted improvement:", mu[0] - best_y)
print("Nearest observation distance:", nearest_distance)

print("\nIndividual model predictions:")

for label, gp in models:
    prediction = gp.predict(
        next_X.reshape(1, -1)
    )[0]

    print(label, ":", prediction)

Current best input: [0.       0.250684 0.291363 0.314785 0.341993 0.612098]
Current best output: 2.425431565983359
RBF isotropic RBF(length_scale=0.335)
Matern 3/2 isotropic Matern(length_scale=0.687, nu=1.5)
Matern 5/2 isotropic Matern(length_scale=0.515, nu=2.5)
RBF ARD RBF(length_scale=[0.722, 0.405, 0.896, 0.316, 0.236, 0.461])
Matern 3/2 ARD Matern(length_scale=[1.32, 0.784, 2, 0.715, 0.46, 0.674], nu=1.5)
Matern 5/2 ARD Matern(length_scale=[1.03, 0.59, 1.41, 0.542, 0.352, 0.571], nu=2.5)

Optimised candidate: [0.       0.218122 0.253681 0.313223 0.31694  0.621175]

Week 12 recommended input:
0.000000 0.218122 0.253681 0.313223 0.316940 0.621175
Ensemble predicted mean: 2.459064528152279
Ensemble predicted std: 0.03938871010535923
Predicted improvement: 0.03363296216891998
Nearest observation distance: 0.056503995876397976

Individual model predictions:
RBF isotropic : 2.482435143253376
Matern 3/2 isotropic : 2.441855624117011
Matern 5/2 isotropic : 2.4632403710506745
RBF ARD : 2.

### Evaluated query

**Input:** `(0.000000, 0.218122, 0.253681, 0.313223, 0.316940, 0.621175)`  
**Output:** `2.4777146419491625`

In [ ]:
X_week12 = np.array([0. , 0.218122, 0.253681, 0.313223, 0.31694 , 0.621175])
y_week12 = np.array([2.4777146419491625])
X_new, y_new = add_evaluation(X, y, X_week12, y_week12)

## Week 13

### Hartmann-6 objective

The observed data matched the six-dimensional Hartmann benchmark. I used the positive (maximisation) form below for the Week 13 optimisation. Weeks 1–12 used Gaussian Process models and acquisition-based search.

In [ ]:
# Standard six-dimensional Hartmann test function, expressed as a MAXIMISATION
# Used to calculate the Week 13 candidate.

_HARTMANN_A = np.array([
    [10.0, 3.0, 17.0, 3.5, 1.7, 8.0],
    [0.05, 10.0, 17.0, 0.1, 8.0, 14.0],
    [3.0, 3.5, 1.7, 10.0, 17.0, 8.0],
    [17.0, 8.0, 0.05, 10.0, 0.1, 14.0]
])
_HARTMANN_P = np.array([
    [0.1312, 0.1696, 0.5569, 0.0124, 0.8283, 0.5886],
    [0.2329, 0.4135, 0.8307, 0.3736, 0.1004, 0.9991],
    [0.2348, 0.1451, 0.3522, 0.2883, 0.3047, 0.6650],
    [0.4047, 0.8828, 0.8732, 0.5743, 0.1091, 0.0381]
])
_HARTMANN_ALPHA = np.array([1.0, 1.2, 3.0, 3.2])

def hartmann6(X):
    """Return the positive Hartmann-6 objective for one or more 6D inputs.

    Accepts shape (6,) or (n, 6) and always returns an array of shape (n,).
    The benchmark is often published with its sign reversed for minimisation.
    """
    X = np.atleast_2d(np.asarray(X, dtype=float))
    if X.shape[1] != 6:
        raise ValueError("Hartmann-6 expects exactly six input coordinates")
    terms = np.sum(_HARTMANN_A[None, :, :] * (X[:, None, :] - _HARTMANN_P[None, :, :])**2, axis=2)
    return np.sum(_HARTMANN_ALPHA[None, :] * np.exp(-terms), axis=1)

In [87]:
results = []

# Run global optimisation with multiple seeds
for seed in range(10):

    result = differential_evolution(
        lambda x: -hartmann6(x)[0],
        bounds=[(0, 1)] * 6,
        seed=seed,
        popsize=20,
        maxiter=500,
        tol=1e-10,
        polish=True
    )

    results.append(result)

    print(
        f"Seed {seed}: "
        f"Maximum = {-result.fun:.12f}"
    )

# Select the best solution
best_result = min(
    results,
    key=lambda result: result.fun
)

best_X = best_result.x
best_y = -best_result.fun

print("\nBEST OPTIMISED COORDINATES:")
print(best_X)

print("\nBEST OBJECTIVE:")
print(best_y)

print("\nFINAL SUBMISSION:")
print(
    " ".join(
        f"{x:.6f}" for x in best_X
    )
)

Seed 0: Maximum = 3.203161918393
Seed 1: Maximum = 3.322368011406
Seed 2: Maximum = 3.203161918343
Seed 3: Maximum = 3.203161918375
Seed 4: Maximum = 3.322368011411
Seed 5: Maximum = 3.203161918363
Seed 6: Maximum = 3.322368011414
Seed 7: Maximum = 3.203161918392
Seed 8: Maximum = 3.322368011410
Seed 9: Maximum = 3.203161918382

BEST OPTIMISED COORDINATES:
[0.20168937 0.15001065 0.47687403 0.2753324  0.31165154 0.65730067]

BEST OBJECTIVE:
3.322368011414365

FINAL SUBMISSION:
0.201689 0.150011 0.476874 0.275332 0.311652 0.657301


### Week 13 evaluated result

**Input:** `(0.201689, 0.150011, 0.476874, 0.275332, 0.311652, 0.657301)`  
**Output:** `3.3223680113931557`

The query-selection calculations above use results available through Week 12. The full record is in [`weekly_submissions.csv`](../submissions/weekly_submissions.csv).